#init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import DateType, StringType
from pyspark.sql.functions import col, trim

In [0]:
df = spark.table('workspace.bronze.erp_loc_a101')

##table preview

In [0]:
df.display()

#Transofrmation

## trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

##customer id cleanup

In [0]:
df = df.withColumn('cid', F.regexp_replace(col('CID'), '-', ''))


## country normalization

In [0]:
df = df.withColumn(
    'cntry',
    F.when(col('cntry') == 'DE', 'Germany')
     .when(col('CNTRY').isin('US', 'USA'), 'United States')
     .when((col('cntry') == '') | (col('cntry').isNull()), 'n/a')
     .otherwise(col('cntry'))
)

##renaming columns

In [0]:
rename_map = {
    'cid': 'customer_number',
    'cntry' : 'country'
}

for old_name, new_name in rename_map.items():
  df = df.withColumnRenamed(old_name, new_name)

## validate dataframe

In [0]:
df.limit(10).display()

#Writing to silver layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customer_location')

## rechecking table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location